In [ ]:
import re
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/dlcourseproject/nlp-getting-started/train.csv")

In [ ]:
# 1. Load dataset
df = df[['text', 'target']].dropna()

In [ ]:
df

,text,target
0,Our Deeds are the Reason of this #earthquake M...,1
1,Forest fire near La Ronge Sask. Canada,1
2,All residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...
7608,Two giant cranes holding a bridge collapse int...,1
7609,@aria_ahrary @TheTawniest The out of control w...,1
7610,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,Police investigating after an e-bike collided ...,1


In [ ]:
# 2. Light cleaning for BERT
def clean_for_bert(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['text'] = df['text'].apply(clean_for_bert)

In [ ]:
# 3. Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['target'],
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

In [ ]:
# 4. Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=96
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=96
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# 5. Dataset class
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TweetDataset(train_encodings, list(y_train))
test_dataset = TweetDataset(test_encodings, list(y_test))

In [ ]:
# 6. Model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# 7. Training args
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    report_to="none"
)

In [ ]:
# 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [ ]:
# 9. Train
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.346284,0.408043
2,0.455079,0.465718
3,0.282397,0.589503


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2286, training_loss=0.35645805068529185, metrics={'train_runtime': 427.4772, 'train_samples_per_second': 42.739, 'train_steps_per_second': 5.348, 'total_flos': 788654832890400.0, 'train_loss': 0.35645805068529185, 'epoch': 3.0})

In [ ]:
# 10. Evaluation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
true = []

for item in test_dataset:
    inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() if k != "labels"}
    label = item['labels'].item()

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    preds.append(pred)
    true.append(label)

bert_acc = accuracy_score(true, preds)
print("BERT Accuracy:", bert_acc)
print(classification_report(true, preds))
print(predict_bert("earthquake destroyed buildings"))
print(predict_bert("flood of emotions after breakup"))
print(predict_bert("this match is fire 🔥"))

BERT Accuracy: 0.8522652659225214
              precision    recall  f1-score   support

           0       0.85      0.90      0.87       869
           1       0.86      0.79      0.82       654

    accuracy                           0.85      1523
   macro avg       0.85      0.84      0.85      1523
weighted avg       0.85      0.85      0.85      1523



In [ ]:
# 11. Save model and tokenizer
model.save_pretrained("/content/drive/MyDrive/dlcourseproject/bert_model")
tokenizer.save_pretrained("/content/drive/MyDrive/dlcourseproject/bert_model")

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!


In [ ]:
def predict_bert(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    pred = probs[0][1].item()

    if pred > 0.7:
        return "Disaster", pred
    elif pred < 0.3:
        return "Not Disaster", pred
    else:
        return "Uncertain", pred

In [ ]:
def predict_multiple(tweets):
    results = []

    for tweet in tweets:
        label, score = predict_bert(tweet)
        results.append({
            "tweet": tweet,
            "prediction": label,
            "confidence": round(score, 4)
        })

    return results

In [ ]:
tweets = [
    "earthquake destroyed buildings",
    "just chilling with friends",
    "flood in city area",
    "this match is fire 🔥"
]

results = predict_multiple(tweets)

for r in results:
    print(r)

{'tweet': 'earthquake destroyed buildings', 'prediction': 'Disaster', 'confidence': 0.9881}
{'tweet': 'just chilling with friends', 'prediction': 'Not Disaster', 'confidence': 0.0586}
{'tweet': 'flood in city area', 'prediction': 'Disaster', 'confidence': 0.9872}
{'tweet': 'this match is fire 🔥', 'prediction': 'Not Disaster', 'confidence': 0.0677}
